# Демо 2. Шаблони розмови і підготовка даних для SFT

В демо 1 донавчання моделі було повʼязане з заміною голови, а отже частковою зміною архітектури нейронів. В багатьох випадках при донавчанні LLM моделей архітектура не змінюється, а змінюється радше формат введених і виведених данних. Для цього потрібно зрозуміти, **у якому вигляді модель бачить діалог**.
В будь-якому випадку фізично на вхід моделі подається тензор з токенами, а за те як з цих токенів утворюються семантичні повідомлення відповідають **шаблони розмови** (chat templates).
Під час донавчання модель може вивчити новий шаблон розмови.

Донавчання LLM методом SFT виконується на діалогах, відформатованих шаблоном розмови.
У цьому ноутбуці розглянемо, що таке шаблон розмови і як підготувати дані у форматі діалогів;
саме донавчання виконаємо в демо 3.

План:
1. Завантажити base- та instruct-версії SmolLM3.
2. Подивитися, як шаблон розмови перетворює різні типи діалогів на текст.
3. Порівняти відповіді base- та instruct-моделі.
4. Перевірити режими міркування `/think` і `/no_think`.
5. Порівняти шаблони різних моделей.
6. Підготувати датасет для SFT у форматі `messages`.

Демо за мотивами вправ курсу [smol-course](https://huggingface.co/learn/smol-course) і розділу 11
[Hugging Face LLM Course](https://huggingface.co/learn/llm-course/chapter11/1).
Для запуску в Colab оберіть середовище з GPU: *Runtime → Change runtime type → T4 GPU*.

In [1]:
# Встановлюємо потрібні бібліотеки; -q вимикає зайвий вивід під час встановлення
!pip install -q "transformers>=4.55.0" "datasets" "accelerate"

In [2]:
import warnings

import torch
from datasets import get_dataset_split_names, load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

if torch.cuda.is_available():
    device = "cuda"
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"Памʼять GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif torch.backends.mps.is_available():
    device = "mps"
    print("Apple MPS")
else:
    device = "cpu"
    warnings.warn("Обчислення на CPU: генерація буде дуже повільною, потрібна GPU")

GPU: Tesla T4
Памʼять GPU: 15.6 GB


In [3]:
# Детальна інформація про GPU NVIDIA: модель, зайнята й вільна памʼять
if device == "cuda":
    !nvidia-smi

Thu Nov 19 14:15:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             11W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Завантаження моделей

[**SmolLM3**](https://huggingface.co/blog/smollm3) — відкрита модель від Hugging Face на 3 млрд параметрів. Вона має дві версії:
- **Base** — лише після попереднього тренування: продовжує текст, але не веде діалог;
- **Instruct** (`SmolLM3-3B`) — донавчена виконувати інструкції й вести діалог.

In [4]:
base_model_name = "HuggingFaceTB/SmolLM3-3B-Base"
instruct_model_name = "HuggingFaceTB/SmolLM3-3B"

base_tokenizer = AutoTokenizer.from_pretrained(base_model_name)
instruct_tokenizer = AutoTokenizer.from_pretrained(instruct_model_name)

# Ваги SmolLM3 збережено в bfloat16, але T4 не підтримує його апаратно,
# тому завантажуємо у float16.
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name, dtype=torch.float16, device_map="auto"
)
instruct_model = AutoModelForCausalLM.from_pretrained(
    instruct_model_name, dtype=torch.float16, device_map="auto"
)
print("Моделі завантажено")

config.json:   0%|          | 0.00/943 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.4k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/151 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

... (12 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/182 [00:00<?, ?B/s]

Моделі завантажено


## 2. Як шаблон розмови форматує діалог

Діалог задається списком повідомлень із ролями:
- `system` — інструкції для моделі (роль, стиль, обмеження);
- `user` — повідомлення користувача;
- `assistant` — відповіді моделі.

Метод `apply_chat_template` перетворює цей список на рядок зі спеціальними токенами.

Параметр `add_generation_prompt`:
- `False` — для передачі в модель завершених діалогів, тобто для **тренувальних даних**;
- `True` — додає в кінець заголовок відповіді асистента, щоб модель почала відповідати; використовується для **інференсу**.

Для SmolLM3 у системне повідомлення можна додати прапорець `/think` або `/no_think`,
який вмикає чи вимикає режим міркування.

In [5]:
# Різні типи діалогів для порівняння
conversations = {
    "simple_qa": [
        {"role": "system", "content": "/no_think"},
        {"role": "user", "content": "What is machine learning?"},
    ],
    "with_system": [
        {
            "role": "system",
            "content": "You are a helpful AI assistant specialized in explaining technical concepts clearly. /no_think",
        },
        {"role": "user", "content": "What is machine learning?"},
    ],
    "multi_turn": [
        {"role": "system", "content": "You are a math tutor. /no_think"},
        {"role": "user", "content": "What is calculus?"},
        {
            "role": "assistant",
            "content": "Calculus is a branch of mathematics that deals with rates of change and accumulation of quantities.",
        },
        {"role": "user", "content": "Can you give me a simple example?"},
    ],
    "reasoning_task": [
        {"role": "system", "content": "/think"},
        {
            "role": "user",
            "content": "Solve step by step: If a train travels 120 miles in 2 hours, what is its average speed?",
        },
    ],
}

for conv_type, messages in conversations.items():
    print(f"--- {conv_type.upper()} ---")

    # Завершений діалог: так виглядають тренувальні дані
    formatted_complete = instruct_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    # Діалог для інференсу: у кінці додано заголовок відповіді асистента
    formatted_prompt = instruct_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    print("Завершений діалог (add_generation_prompt=False):")
    print(formatted_complete)
    print("\nДля інференсу (add_generation_prompt=True):")
    print(formatted_prompt)
    print("\n" + "=" * 50 + "\n")

--- SIMPLE_QA ---
Завершений діалог (add_generation_prompt=False):
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 19 November 2026
Reasoning Mode: /no_think

## Custom Instructions

You are a helpful AI assistant named SmolLM, trained by Hugging Face.

<|im_start|>user
What is machine learning?<|im_end|>


Для інференсу (add_generation_prompt=True):
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 19 November 2026
Reasoning Mode: /no_think

## Custom Instructions

You are a helpful AI assistant named SmolLM, trained by Hugging Face.

<|im_start|>user
What is machine learning?<|im_end|>
<|im_start|>assistant
<think>

</think>



--- WITH_SYSTEM ---
Завершений діалог (add_generation_prompt=False):
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 19 November 2026
Reasoning Mode: /no_think

## Custom Instructions

You are a helpful AI assistant specialized in explaining technical concepts clearly.

<|im_s

**Що варто помітити:**
- Шаблон SmolLM3 сам додає до системного повідомлення блок метаданих (дата відсічення знань, поточна дата, режим міркування)
  і стандартну інструкцію, якщо власної немає. Модель бачить більше, ніж ми написали.
- Кожне повідомлення обгорнуте токенами `<|im_start|>роль` … `<|im_end|>` (формат ChatML).
- З `add_generation_prompt=True` рядок закінчується на `<|im_start|>assistant`: з цього місця модель починає генерувати.

## 3. Base проти Instruct

Ставимо обом моделям одне питання. Base-модель отримує сирий текст без шаблону,
instruct-модель — діалог, відформатований шаблоном розмови.

In [6]:
def generate(model, tokenizer, text, max_new_tokens=150, temperature=0.7):
    """Генерує продовження тексту й повертає лише нові токени, без промпту."""
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    # Відкидаємо токени промпту: декодуємо лише те, що згенерувала модель
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

In [7]:
test_prompt = "Explain me, what is quantum computing?"

# Base-модель: сирий текст без шаблону
print("🤖 BASE-МОДЕЛЬ:")
print("-" * 50)
print(generate(base_model, base_tokenizer, test_prompt))

# Instruct-модель: діалог, відформатований шаблоном розмови
instruct_messages = [
    {"role": "system", "content": "/no_think"},
    {"role": "user", "content": test_prompt},
]
instruct_formatted = instruct_tokenizer.apply_chat_template(
    instruct_messages, tokenize=False, add_generation_prompt=True
)
print("\n" + "=" * 50)
print("💬 INSTRUCT-МОДЕЛЬ:")
print("-" * 50)
print(generate(instruct_model, instruct_tokenizer, instruct_formatted))

🤖 BASE-МОДЕЛЬ:
--------------------------------------------------


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


 How did it came about?
A quantum computer is a computer with the power to perform multiple calculations at the same time, not just multiple calculations sequentially.
Yes, I can answer that for you. It is a computer based on quantum mechanics. Just like digital computers rely on the properties of electrons in semiconductors, quantum computers rely on the properties of electrons in atoms.
Atoms have an electron cloud. The electrons in the cloud can be in a variety of states. A quantum computer is designed to take advantage of these states to perform calculations that are impossible to do on a digital computer.

💬 INSTRUCT-МОДЕЛЬ:
--------------------------------------------------
Quantum computing is a revolutionary field of computing that leverages the principles of quantum mechanics to perform calculations and manipulate data in ways that are fundamentally different from classical computers. Here's a simplified explanation:

**What is Quantum Mechanics?**

Quantum mechanics is a bran

## 4. Режими міркування `/think` і `/no_think`

SmolLM3 — гібридна модель: у режимі `/think` вона спочатку генерує міркування в блоці `<think>…</think>`,
а потім відповідь; у режимі `/no_think` блок міркувань порожній.

In [22]:
reasoning_prompts = [
    "What is 15 × 24? Show your work.",
    "A recipe calls for 2 cups of flour for 12 cookies. How much flour is needed for 30 cookies?",
    "If I have $50 and spend $18.75 on lunch and $12.30 on a book, how much money do I have left?",
]

for thinking_flag in ["/no_think", "/think"]:
    print(f"===== Режим: {thinking_flag} =====\n")
    for i, prompt in enumerate(reasoning_prompts, 1):
        messages = [
            {"role": "system", "content": thinking_flag},
            {"role": "user", "content": prompt},
        ]
        formatted = instruct_tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        # Нижча температура робить відповіді на арифметичні задачі стабільнішими
        answer = generate(instruct_model, instruct_tokenizer, formatted, max_new_tokens=300, temperature=0.3)
        print(f"Задача {i}: {prompt}")
        print(f"Відповідь: {answer}")
        print("\n" + "-" * 50 + "\n")

===== Режим: /no_think =====

Задача 1: What is 15 × 24? Show your work.
Відповідь: To solve the multiplication problem 15 × 24, we can break it down step by step.

First, let's multiply 15 by 4, which is the ones place of 24:
15 × 4 = 60

Next, we multiply 15 by 20, which is the tens place of 24:
15 × 20 = 300

Now, we add the two results together:
60 + 300 = 360

So, the answer to 15 × 24 is 360.

Alternatively, we can also use the standard multiplication algorithm:

```
   15
× 24
----
   60  (15 × 4)
+300   (15 × 20)
----
  360
```

Both methods yield the same result: 360.

--------------------------------------------------

Задача 2: A recipe calls for 2 cups of flour for 12 cookies. How much flour is needed for 30 cookies?
Відповідь: To determine how much flour is needed for 30 cookies, we can start by finding out how much flour is required for one cookie. 

The recipe calls for 2 cups of flour for 12 cookies. To find the amount of flour per cookie, we divide the total amount of 

## 5. У кожної моделі свій шаблон

Ролі `system`, `user`, `assistant` — це домовленість на рівні формату `messages`.
У токенах кожна модель записує їх по-своєму. Порівняємо шаблони кількох моделей.

In [23]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hi!"},
    {"role": "assistant", "content": "Hello! How can I help?"},
]

for name in ["HuggingFaceTB/SmolLM3-3B", "Qwen/Qwen3-0.6B", "microsoft/Phi-3.5-mini-instruct"]:
    tok = AutoTokenizer.from_pretrained(name)
    print(f"===== {name} =====")
    print(tok.apply_chat_template(messages, tokenize=False))
    print()

===== HuggingFaceTB/SmolLM3-3B =====
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 19 November 2026
Reasoning Mode: /think

## Custom Instructions

You are a helpful assistant.

<|im_start|>user
Hi!<|im_end|>
<|im_start|>assistant
Hello! How can I help?<|im_end|>


===== Qwen/Qwen3-0.6B =====
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Hi!<|im_end|>
<|im_start|>assistant
<think>

</think>

Hello! How can I help?<|im_end|>


===== microsoft/Phi-3.5-mini-instruct =====
<|system|>
You are a helpful assistant.<|end|>
<|user|>
Hi!<|end|>
<|assistant|>
Hello! How can I help?<|end|>
<|endoftext|>



Прапорці режиму міркування також залежать від моделі. У Qwen3, наприклад,
режим вмикається параметром `enable_thinking` у `apply_chat_template`.

In [16]:
qwen_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
question = [{"role": "user", "content": "What is 2 + 2?"}]

for enable_thinking in [True, False]:
    print(f"===== enable_thinking={enable_thinking} =====")
    print(qwen_tokenizer.apply_chat_template(
        question, tokenize=False, add_generation_prompt=True, enable_thinking=enable_thinking
    ))

===== enable_thinking=True =====
<|im_start|>user
What is 2 + 2?<|im_end|>
<|im_start|>assistant

===== enable_thinking=False =====
<|im_start|>user
What is 2 + 2?<|im_end|>
<|im_start|>assistant
<think>

</think>




### А що з base-моделлю?

Base-модель не донавчали на діалогах, тому шаблон у її токенізаторі або відсутній, або модель ніколи не бачила його токенів у ролі розмітки діалогу.
Перед SFT шаблон для base-моделі задають самостійно — найпростіше взяти його з instruct-версії тієї самої моделі.

In [17]:
print("Шаблон є в base-токенізаторі:  ", base_tokenizer.chat_template is not None)
print("Шаблон є в instruct-токенізаторі:", instruct_tokenizer.chat_template is not None)

Шаблон є в base-токенізаторі:   False
Шаблон є в instruct-токенізаторі: True


## 6. Підготовка датасету для SFT

Для SFT дані зберігають у форматі `messages` — тобто у списку повідомлень із ролями.
Бібліотека TRL (`SFTTrainer`) сама застосовує до нього шаблон розмови під час тренування.

### SmolTalk2

**SmolTalk2** — датасет, на якому донавчали SmolLM3. Він складається з багатьох частин (splits) з різних джерел.
Список частин можна отримати без завантаження самих даних.

In [18]:
splits = get_dataset_split_names("HuggingFaceTB/smoltalk2", "SFT")
print(f"Кількість частин: {len(splits)}")
for s in splits:
    print(" -", s)

Resolving data files:   0%|          | 0/124 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

Кількість частин: 25
 - LongAlign_64k_Qwen3_32B_yarn_131k_think
 - OpenThoughts3_1.2M_think
 - aya_dataset_Qwen3_32B_think
 - multi_turn_reasoning_if_think
 - s1k_1.1_think
 - smolagents_toolcalling_traces_think
 - smoltalk_multilingual8_Qwen3_32B_think
 - smoltalk_systemchats_Qwen3_32B_think
 - table_gpt_Qwen3_32B_think
 - LongAlign_64k_context_lang_annotated_lang_6_no_think
 - Mixture_of_Thoughts_science_no_think
 - OpenHermes_2.5_no_think
 - OpenThoughts3_1.2M_no_think_no_think
 - hermes_function_calling_v1_no_think
 - smoltalk_multilingual_8languages_lang_5_no_think
 - smoltalk_smollm3_everyday_conversations_no_think
 - smoltalk_smollm3_explore_instruct_rewriting_no_think
 - smoltalk_smollm3_smol_magpie_ultra_no_think
 - smoltalk_smollm3_smol_rewrite_no_think
 - smoltalk_smollm3_smol_summarize_no_think
 - smoltalk_smollm3_systemchats_30k_no_think
 - table_gpt_no_think
 - tulu_3_sft_personas_instruction_following_no_think
 - xlam_traces_no_think
 - smoltalk_everyday_convs_reasoning_

In [19]:
# Завантажуємо лише файл однієї частини через data_files: повний датасет займає десятки гігабайт,
# а load_dataset(..., split=...) спершу перебирає файли всіх частин і працює дуже довго
split = "smoltalk_everyday_convs_reasoning_Qwen3_32B_think"
everyday = load_dataset("HuggingFaceTB/smoltalk2", data_files=f"SFT/{split}-*.parquet", split="train")
print(everyday)
print()
for message in everyday[0]["messages"]:
    print(f"[{message['role']}] {message['content'][:200]}")

Dataset({
    features: ['messages', 'chat_template_kwargs', 'source'],
    num_rows: 2057
})

[user] Hi there
[assistant] <think>
Okay, the user sent "Hi there". That's a friendly greeting. I should respond in a welcoming way. Let me check the guidelines. I need to be helpful, keep the conversation going, and maybe ask h
[user] I'm looking for a healthy breakfast idea. What's a good option?
[assistant] <think>
Okay, the user is asking for a healthy breakfast idea. Let me think about what makes a breakfast healthy. It should be balanced, providing a mix of nutrients like protein, fiber, healthy fats,


Ця частина зібрана для режиму `/think`, тому відповіді асистента містять блок міркувань `<think>…</think>`.

Як видно з назви частини, відповіді в ній згенерувала модель Qwen3-32B. Такі **синтетичні дані** від сильнішої моделі —
поширений спосіб отримати тисячі прикладів для SFT без ручного написання відповідей. Помилки моделі-генератора при цьому переходять у дані,
тому синтетичні дані фільтрують.

### Перетворення власного датасету у формат `messages`

Більшість датасетів має інший формат, наприклад колонки «питання» і «відповідь».
Перетворимо датасет задач з математики **GSM8K** у формат `messages`.

In [20]:
gsm8k = load_dataset("openai/gsm8k", "main", split="train[:100]")  # невелика вибірка для демо
print("Оригінальний приклад:", gsm8k[0])


def to_messages(examples):
    """Перетворює пари «питання — відповідь» на діалоги з ролями."""
    processed = []
    for question, answer in zip(examples["question"], examples["answer"]):
        processed.append([
            {"role": "system", "content": "You are a math tutor. Solve problems step by step."},
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer},
        ])
    return {"messages": processed}


gsm8k_messages = gsm8k.map(to_messages, batched=True, remove_columns=gsm8k.column_names)
print("\nУ форматі messages:", gsm8k_messages[0])

Оригінальний приклад: {'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}

У форматі messages: {'messages': [{'content': 'You are a math tutor. Solve problems step by step.', 'role': 'system'}, {'content': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'role': 'user'}, {'content': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72', 'role': 'assistant'}]}


### Що побачить модель під час тренування

Застосуємо шаблон розмови до підготовленого прикладу. Саме цей рядок модель отримає під час SFT.

In [21]:
text = instruct_tokenizer.apply_chat_template(
    gsm8k_messages[0]["messages"],
    tokenize=False,
    add_generation_prompt=False,  # діалог завершений: відповідь асистента вже є
)
print(text)
print(f"Довжина в токенах: {len(instruct_tokenizer(text)['input_ids'])}")

<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 19 November 2026
Reasoning Mode: /think

## Custom Instructions

You are a math tutor. Solve problems step by step.

<|im_start|>user
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?<|im_end|>
<|im_start|>assistant
Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72<|im_end|>

Довжина в токенах: 145


**Що варто помітити:** шаблон знову додав метадані й режим `/think`, хоча в даних їх немає,
а відповідь асистента не містить блоку міркувань. Якщо донавчати на таких прикладах, модель вчитиметься
відповідати без міркувань навіть у режимі `/think`. Тому формат даних і шаблон мають узгоджуватися з тим,
як модель використовуватиметься після донавчання.

## Відповіді на питання до демо 2

**11. Час виконання та проблеми із запуском.**
Увесь ноутбук на T4 відпрацював приблизно за 10–11 хвилин. Найдовше (≈4 хв) тривало завантаження двох моделей SmolLM3-3B (Base та Instruct, кожна ≈6 ГБ у fp16). Генерація base проти instruct зайняла 12 с, секція 4 із шістьма відповідями по 300 токенів — 56 с, решта комірок — секунди. Проблеми були такі:
- Colab спершу не давав підключитися до GPU: «Забагато сеансів» (безкоштовний акаунт дозволяє лише одну GPU-сесію, тож я завершив сесію демо 1), а потім тимчасово «немає доступної серверної частини з GPU», і підключитися вдалося лише згодом.
- Під час першого «Виконати все» виводи секцій 4–5 не відобразилися в ноутбуці, тому ці комірки я запустив повторно.
- Є попередження про неавторизовані запити до HF Hub (без HF_TOKEN) і про clean_up_tokenization_spaces. Вони не впливають на результат, лише на ліміти й швидкість завантаження.

**12. Навіщо змінювати add_generation_prompt і як визначити його значення за виводом.**
Параметр показує, що має бути в кінці рядка. З False шаблон форматує діалог «як є»: це потрібно для тренувальних прикладів, де остання репліка вже є відповіддю асистента і модель вчиться на повному тексті. З True шаблон дописує на кінець заголовок відповіді асистента, щоб модель почала генерувати саме відповідь, а не продовжувала репліку користувача (для інференсу).
За виводом секції 2 це видно одразу. З False рядок закінчується на `<|im_end|>` останнього повідомлення. З True він закінчується на `<|im_start|>assistant` (у режимі /no_think після нього ще порожній блок think). Тобто в нашому діалозі, що закінчується повідомленням користувача, True додає порожнє місце для відповіді. Правило таке: якщо останнє повідомлення — assistant (приклад для навчання), ставлю False; якщо останнє — user і я чекаю відповідь, ставлю True.

**13. Різниця між ролями system і assistant.**
system — це інструкції розробника: роль, стиль, обмеження та режим міркування. Вони задаються один раз на початку діалогу, а шаблон SmolLM3 вкладає їх у блок Custom Instructions разом із метаданими. assistant — це репліки самої моделі. Під час інференсу їх генерує модель, а під час SFT саме на цих токенах рахується loss. У multi_turn попередня відповідь асистента потрапила в історію як контекст, і шаблон додав у неї порожній блок think, бо режим /no_think.

**14. Коментар до виводу секції 2.**
Шаблон SmolLM3 сам додав і перетворив чимало з того, що я не писав:
- на початок системного повідомлення додано блок Metadata: Knowledge Cutoff Date (June 2025), Today Date (сьогоднішня дата запуску, 19 November 2026) і Reasoning Mode. Мій прапорець /think або /no_think з тексту прибрано і перенесено в рядок Reasoning Mode;
- у Custom Instructions потрапив мій текст. Коли свого system не було (simple_qa), підставилася стандартна фраза про SmolLM від Hugging Face, а для /think — довга стандартна інструкція про структуру Thought/Solution;
- кожне повідомлення обгорнуто ChatML-токенами <|im_start|>роль … `<|im_end|>`;
- з add_generation_prompt=True додано `<|im_start|>assistant`, а в /no_think ще й порожній блок think, щоб модель не міркувала. У /think цього блоку немає: модель сама його відкриє.
Навіщо це потрібно: модель завжди бачить однаковий формат, знає дату та режим і чітко розрізняє ролі.

**15. Base проти Instruct (секція 3).**
Base-модель отримала сирий текст без шаблону і не відповіла на питання, а продовжила текст: почала зі шматка якогось іншого речення («How did it came about?», до того ж із граматичною помилкою), далі змішала фрази питання та відповіді, а в поясненні є неточності. Це типове продовження документа. Instruct-модель із шаблоном відповіла як асистент: впевнено, структуровано, із заголовками «What is Quantum Mechanics?», «What are Qubits?»; текст обірвався на півслові лише через ліміт 150 токенів. Різниця не в обсязі знань, а в тому, що Instruct донавчали (SFT) відповідати на інструкції у форматі чату, а base тільки передбачає наступні токени. Генерація випадкова (do_sample=True, temperature=0.7), тож при повторному запуску текст буде іншим, але характер відповідей збережеться.

**16. Чому max_new_tokens=300 у режимі /think.**
У /think модель спочатку пише довге міркування в блоці think, і лише потім відповідь, а всі ці токени рахуються в ліміті. Тому для міркувань потрібен більший запас, ніж у /no_think. Але мій запуск показав, що навіть 300 токенів замало: у всіх трьох задачах відповідь у /think обірвалася всередині міркування, до закриття блоку think і до фінального розв'язку не дійшло (у задачі 1 модель уже отримала 360 і писала «Wait, is that right?», у задачі 3 уже отримала 18.95 і перевіряла себе). Тобто 300 — це компроміс між часом генерації та довжиною. Для реального використання /think потрібно ставити ліміт у кілька сотень або й понад тисячу токенів.

**17. Чи впоралася модель із математичними задачами і який режим я б обрав.**
У режимі /no_think усі три задачі розв'язано правильно і з поясненням по кроках: 15 × 24 = 360, для 30 печив потрібно 5 склянок борошна, залишок грошей — $18.95. У режимі /think міркування теж були правильними за ходом (60 + 300 = 360; пропорція 2·30 = 12x; 50 − 18.75 = 31.25, далі 18.95), із самоперевіркою на кшталт «Wait, is that right?». Але через ліміт у 300 токенів жодна відповідь у /think не дійшла до кінця, тож готового розв'язку я не отримав. Для таких простих арифметичних задач я б обрав /no_think: він швидший, дешевший за токени, а результат правильний. /think має сенс для складних багатокрокових задач, де потрібна самоперевірка, і лише з великим max_new_tokens та готовністю чекати довше.

**18. Відмінності шаблонів у секції 5.**
Ролі system/user/assistant однакові, але записуються по-різному:
- SmolLM3: токени `<|im_start|>` і `<|im_end|>`, а ще блок Metadata з датою, Knowledge Cutoff і Reasoning Mode та розділ Custom Instructions;
- Qwen3-0.6B: теж `<|im_start|>` і `<|im_end|>`, але без метаданих, system вставлено як є. У відповідь асистента в історії шаблон додав порожній блок think;
- Phi-3.5-mini: зовсім інші токени `<|system|>`, `<|user|>`, `<|assistant|>`, кожне повідомлення закривається `<|end|>`, а в кінці стоїть `<|endoftext|>`.
Режим міркування теж залежить від моделі. У SmolLM3 його вмикає прапорець /think або /no_think у системному повідомленні. У Qwen3 для цього є параметр enable_thinking: з True після `<|im_start|>`assistant нічого не додається (модель відкриє think сама), а з False шаблон одразу вставляє порожній блок think. Тому шаблон треба брати саме від тієї моделі, яку я донавчаю.

**19. Навіщо знати шаблони розмови для SFT.**
Модель навчається на конкретній послідовності токенів, і шаблон визначає, як у ній записані ролі, кінці реплік та режим міркування. Якщо формат під час SFT відрізняється від того, що буде під час інференсу, модель бачить «чужі» токени, плутає ролі й не вчиться вчасно закінчувати відповідь (`<|im_end|>`). У base-моделі шаблона немає (у виводі «Шаблон є в base-токенізаторі: False», а в instruct — True), тож його доведеться задати самому, наприклад узявши з instruct-версії. Крім того, шаблон додає частину тексту автоматично (метадані, порожній think), і це потрапляє в тренувальні дані: я бачу це в прикладі GSM8K, де довжина становить 145 токенів. Також від шаблону залежить, які токени рахуються в loss при assistant_only_loss.

**20. З чого побудовано smoltalk2.**
Це датасет, на якому донавчали SmolLM3. У конфігурації SFT він має 25 частин (splits), зібраних із різних джерел: відкритих наборів та синтетичних діалогів, які згенерували сильніші моделі. Частини для режиму /think мають суфікс _think: це, зокрема, OpenThoughts3_1.2M, s1k_1.1, multi_turn_reasoning_if, smolagents_toolcalling_traces, LongAlign_64k та набори, де відповіді з міркуваннями згенерувала Qwen3-32B (aya_dataset, smoltalk_multilingual8, smoltalk_systemchats, table_gpt). Частини для /no_think мають суфікс _no_think: OpenHermes_2.5, Mixture_of_Thoughts_science, hermes_function_calling_v1, tulu_3_sft_personas_instruction_following і власні набори smoltalk_smollm3_* (everyday_conversations, smol_magpie_ultra, smol_rewrite, smol_summarize, systemchats_30k тощо), а також table_gpt та мультимовні набори. Я завантажив частину smoltalk_everyday_convs_reasoning_Qwen3_32B_think: у ній 2057 прикладів із колонками messages, chat_template_kwargs та source.

**21. Які зміни внесено в дані в розділі «Перетворення власного датасету у формат messages».**
Спочатку GSM8K мав колонки question і answer, причому answer містить розв'язок із калькуляторними вставками `<<48/2=24>>` та рядком #### 72. Функція to_messages перетворює кожен рядок на діалог із трьох повідомлень: system («You are a math tutor. Solve problems step by step.»), user (питання) та assistant (відповідь без змін). Старі колонки видалено (remove_columns), у результаті є одна колонка messages. Для демо взято лише перші 100 прикладів. Тобто новою є тільки системна інструкція, якої в оригіналі не було; текст питань і відповідей не змінено, тому модель вчитиметься відтворювати і формат GSM8K із анотаціями та «####». Після застосування шаблону приклад має 145 токенів. Шаблон ще дописав метадані та Reasoning Mode: /think, хоча відповідь не містить міркувань. Це розбіжність між даними та режимом, яку під час справжнього SFT варто усунути (наприклад, додати /no_think у system).